# Fusion Evaluation Notebook - Music

Load fused results and test set, manually align IDs, then evaluate with type-aware rules.

In [1]:
import pandas as pd
import json
from pathlib import Path

# Add PyDI to path
import sys
sys.path.insert(0, str(Path.cwd().parent.parent.parent))

## 1. Load Data

In [2]:
# Configuration - update these paths
OUTPUT_DIR = Path(".")
FUSION_DIR = OUTPUT_DIR / "fusion"
TEST_DIR = Path("../../../usecases/input/music/fusion")

# Load fused data (from best case or specify path)
best_case_path = FUSION_DIR / "optimization" / "best_case.json"
if best_case_path.exists():
    with open(best_case_path) as f:
        best_info = json.load(f)
    best_case_dir = Path(best_info["best_case_dir"])
    print(f"Best case: {best_info['best_case_key']} (accuracy: {best_info['best_accuracy']:.1%})")
else:
    # Fallback to fused_clean.csv
    best_case_dir = FUSION_DIR

# Load fused data
fused_path = best_case_dir / "fused.csv"
if not fused_path.exists():
    fused_path = FUSION_DIR / "fused_clean.csv"
    
fused_df = pd.read_csv(fused_path)
print(f"Loaded fused data: {len(fused_df)} rows from {fused_path}")
fused_df.head()

KeyError: 'best_accuracy'

In [ ]:
# Load test set
from PyDI.io.loaders import load_xml
from PyDI.normalization import load_normalization_spec
from PyDI.normalization.transform import transform_dataframe

test_xml = TEST_DIR / "test_set.xml"
if test_xml.exists():
    test_df = load_xml(test_xml, nested_handling="aggregate")
    print(f"Loaded test set: {len(test_df)} rows from {test_xml}")
else:
    # Try CSV
    test_csv = TEST_DIR / "test_set.csv"
    test_df = pd.read_csv(test_csv)
    print(f"Loaded test set: {len(test_df)} rows from {test_csv}")

# Apply normalization (same as pipeline Step 1)
SCHEMA_PATH = Path("../../../usecases/input/music/schemamatching/target_schema.json")
SCHEMA_MATCHING_DIR = OUTPUT_DIR / "schema_matching"  # For taxonomy cache

with open(SCHEMA_PATH) as f:
    target_schema = json.load(f)

spec = load_normalization_spec(target_schema)
result = transform_dataframe(
    test_df,
    spec,
    chat_model=None,  # Use cached taxonomy mappings only
    taxonomy_cache_dir=str(SCHEMA_MATCHING_DIR),
    schema_base_path=str(SCHEMA_PATH.parent),
)
test_df = result.dataframe
print(f"Normalized test set: {len(test_df)} records")

test_df.head()

/Users/aaronsteiner/Documents/GitHub/PyDI/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Column 'genre' not found in DataFrame


Loaded test set: 23 rows from ../../../usecases/input/music/fusion/test_set.xml
Normalized test set: 23 records


,id,name_provenance,name,artist_provenance,artist,release-date_provenance,release-date,release-country_provenance,release-country,duration_provenance,duration,label_provenance,label,genre_provenance,tracks_provenance,tracks_track-name
0,mbrainz_6891,lastFM_14591,Ascension Side,mbrainz_6891,Pep Love,mbrainz_6891+discogs_30446,2003-01-01,mbrainz_6891,United States of America,discogs_30446,2798,discogs_30446,Hiero Imperium,,lastFM_14591+mbrainz_6891+discogs_30446,"[Chance, Sabotage, Sabotage, Warrior Poets, Re..."
1,mbrainz_5095,lastFM_10595,Loves Are Like Empires,lastFM_10595,Glissando,mbrainz_5095,2007-01-01,mbrainz_5095,United Kingdom of Great Britain and Northern I...,lastFM_10595,4313,,NaN,,mbrainz_5095+lastFM_10595,"[...and Before We Knew It Had Happened At All,..."
2,mbrainz_1834,mbrainz_1834,Beyond the Gates,mbrainz_1834,Cans,mbrainz_1834,2004-04-26,mbrainz_1834,Finland,mbrainz_1834,3080,discogs_8457,Noise International,,mbrainz_1834+discogs_8457,"[Fields of Yesterday, Soul Collector, Red Ligh..."
3,mbrainz_2222,discogs_9336,Guns At Dawn,discogs_9336,Baron|Pendulum (3),mbrainz_2222+discogs_9336,2005-04-18,mbrainz_2222,United Kingdom of Great Britain and Northern I...,mbrainz_2222,614,discogs_9336,Breakbeat Kaos,,mbrainz_2222+discogs_9336,"[Guns at Dawn, Ratpack]"
4,mbrainz_25432,mbrainz_25432,Up in Dreamland,mbrainz_25432,"Romani, Graziano",mbrainz_25432,2003-10-01,mbrainz_25432,Italy,mbrainz_25432,3636,,NaN,,mbrainz_25432,"[Let's Come Alive, Face the World, Every Road ..."


## 2. Inspect IDs

In [ ]:
print("Fused IDs (first 20):")
print(fused_df["id"].head(20).tolist())
print(f"\nFused ID dtype: {fused_df['id'].dtype}")
print(f"Fused ID nulls: {fused_df['id'].isna().sum()}")

Fused IDs (first 20):
['lastFM_33277', 'discogs_41199', 'lastFM_5200', 'lastFM_22837', 'mbrainz_3085', 'lastFM_4882', 'discogs_23227', 'discogs_24471', 'lastFM_79006', 'discogs_95068', 'discogs_17875', 'lastFM_50291', 'lastFM_51628', 'mbrainz_21338', 'mbrainz_1653', 'mbrainz_9690', 'discogs_51610', 'lastFM_7632', 'mbrainz_3854', 'lastFM_64980']

Fused ID dtype: object
Fused ID nulls: 0


In [ ]:
print("Test set IDs (first 20):")
print(test_df["id"].head(20).tolist())
print(f"\nTest ID dtype: {test_df['id'].dtype}")
print(f"Test ID nulls: {test_df['id'].isna().sum()}")

Test set IDs (first 20):
['mbrainz_6891', 'mbrainz_5095', 'mbrainz_1834', 'mbrainz_2222', 'mbrainz_25432', 'mbrainz_2807', 'mbrainz_9448', 'mbrainz_10613', 'mbrainz_15458', 'mbrainz_3787', 'mbrainz_20856', 'mbrainz_12766', 'mbrainz_3388', 'mbrainz_13699', 'mbrainz_5463', 'mbrainz_15029', 'mbrainz_29595', 'mbrainz_8963', 'mbrainz_159', 'mbrainz_9858']

Test ID dtype: object
Test ID nulls: 0


In [ ]:
# Check overlap
fused_ids = set(fused_df["id"].dropna().astype(str))
test_ids = set(test_df["id"].dropna().astype(str))

overlap = fused_ids & test_ids
print(f"Fused IDs: {len(fused_ids)}")
print(f"Test IDs: {len(test_ids)}")
print(f"Overlap: {len(overlap)}")
print(f"\nSample fused IDs not in test: {list(fused_ids - test_ids)[:5]}")
print(f"Sample test IDs not in fused: {list(test_ids - fused_ids)[:5]}")

Fused IDs: 30885
Test IDs: 23
Overlap: 8

Sample fused IDs not in test: ['discogs_81139', 'discogs_81596', 'discogs_6469', 'discogs_178797', 'lastFM_61791']
Sample test IDs not in fused: ['mbrainz_15029', 'mbrainz_6891', 'mbrainz_29595', 'mbrainz_6947', 'mbrainz_2807']


## 3. ID Alignment

Extract MusicBrainz IDs from `_fusion_sources` to align with test set.

In [ ]:
import ast

def extract_mbrainz_id(fusion_sources):
    """Extract MusicBrainz ID from _fusion_sources list."""
    if pd.isna(fusion_sources) or fusion_sources is None:
        return None
    
    # Parse if it's a string representation of a list
    if isinstance(fusion_sources, str):
        try:
            sources = ast.literal_eval(fusion_sources)
        except (ValueError, SyntaxError):
            sources = [fusion_sources]
    else:
        sources = fusion_sources
    
    # Find the MusicBrainz ID (starts with "mbrainz_")
    for src in sources:
        if isinstance(src, str) and src.startswith("mbrainz_"):
            return src
    
    return None

# Extract MusicBrainz ID from _fusion_sources and use as the main ID
fused_df["id"] = fused_df["_fusion_sources"].apply(extract_mbrainz_id)

# Show how many records now have MusicBrainz IDs
print(f"Records with MusicBrainz ID: {fused_df['id'].notna().sum()} / {len(fused_df)}")
print(f"\nSample IDs after extraction:")
print(fused_df["id"].dropna().head(10).tolist())

Records with MusicBrainz ID: 4717 / 30885

Sample IDs after extraction:
['mbrainz_15126', 'mbrainz_2140', 'mbrainz_2436', 'mbrainz_10678', 'mbrainz_3085', 'mbrainz_2261', 'mbrainz_5364', 'mbrainz_5624', 'mbrainz_31762', 'mbrainz_2507']


In [ ]:
# Verify cleanup - re-check overlap
fused_ids = set(fused_df["id"].dropna().astype(str))
test_ids = set(test_df["id"].dropna().astype(str))
overlap = fused_ids & test_ids

print(f"After cleanup:")
print(f"  Overlap: {len(overlap)} / {len(test_ids)} test records")

After cleanup:
  Overlap: 23 / 23 test records


## 4. Evaluation Functions

Type-aware matching rules:
- **Strings**: Tokenized match (word overlap)
- **Dates**: Year-only comparison
- **Numbers**: 20% relative tolerance
- **Lists**: Set equality

In [ ]:
from PyDI.fusion.evaluation import tokenized_match, year_only_match, set_equality_match
import re
import numpy as np

def _is_null(val) -> bool:
    """Check if value is null/missing, handling arrays safely."""
    if val is None:
        return True
    try:
        result = pd.isna(val)
        if isinstance(result, (bool, np.bool_)):
            return bool(result)
        return False  # If array-like, not a simple null
    except (ValueError, TypeError):
        return False


def numeric_tolerance_match_relative(fused_value, expected_value, tolerance: float = 0.1) -> bool:
    """Numeric tolerance match with 10% relative tolerance (relative to expected)."""
    if fused_value is None or expected_value is None:
        return fused_value is None and expected_value is None
    
    if _is_null(fused_value) or _is_null(expected_value):
        return _is_null(fused_value) and _is_null(expected_value)
    
    try:
        fused_num = float(str(fused_value).replace(",", "").replace("$", "").strip())
        expected_num = float(str(expected_value).replace(",", "").replace("$", "").strip())
    except (ValueError, TypeError):
        return str(fused_value).strip() == str(expected_value).strip()
    
    if expected_num == 0:
        return abs(fused_num) < 1e-9
    
    relative_diff = abs(fused_num - expected_num) / abs(expected_num)
    return relative_diff <= tolerance


def infer_type(value) -> str:
    """Infer type from a value: list, date, numeric, or string."""
    if _is_null(value):
        return "string"
    
    # If it's already a list/array, return list type
    if isinstance(value, (list, tuple, np.ndarray)):
        return "list"
    
    s = str(value).strip()
    
    # Check for list
    if s.startswith("[") or ";" in s or "|" in s:
        return "list"
    
    # Check for date (yyyy-mm-dd or contains year pattern)
    if re.match(r"^\d{4}[-/]\d{1,2}[-/]\d{1,2}", s) or re.match(r"^\d{4}$", s):
        return "date"
    
    # Check for numeric
    try:
        float(s.replace(",", "").replace("$", "").replace("%", ""))
        return "numeric"
    except ValueError:
        pass
    
    return "string"


def get_match_function(value_type: str):
    """Get the appropriate match function for a value type."""
    if value_type == "list":
        return set_equality_match
    elif value_type == "date":
        return year_only_match
    elif value_type == "numeric":
        return numeric_tolerance_match_relative
    else:
        return tokenized_match


print("Evaluation functions loaded.")

Evaluation functions loaded.


## 5. Run Evaluation

In [ ]:
def evaluate_fusion(fused_df, test_df, id_column="id", skip_columns=None):
    """Evaluate fused data against test set using type-aware matching."""
    skip_columns = skip_columns or [id_column, "_fusion_source_datasets", "_fusion_confidence", "_fusion_metadata"]
    
    # Get common attributes
    fused_attrs = set(fused_df.columns) - set(skip_columns)
    test_attrs = set(test_df.columns) - set(skip_columns)
    common_attrs = fused_attrs & test_attrs
    
    print(f"Common attributes: {sorted(common_attrs)}")
    print(f"Fused-only: {sorted(fused_attrs - test_attrs)}")
    print(f"Test-only: {sorted(test_attrs - fused_attrs)}")
    print()
    
    # Build lookup
    fused_lookup = {str(row[id_column]): row for _, row in fused_df.iterrows() if pd.notna(row[id_column])}
    
    results = {
        "total": 0,
        "correct": 0,
        "per_attribute": {attr: {"total": 0, "correct": 0} for attr in common_attrs},
        "mismatches": [],
    }
    
    for _, test_row in test_df.iterrows():
        test_id = str(test_row[id_column])
        fused_row = fused_lookup.get(test_id)
        
        if fused_row is None:
            continue  # Skip if no matching fused record
        
        for attr in common_attrs:
            expected = test_row.get(attr)
            fused = fused_row.get(attr)
            
            # Skip if both null
            if _is_null(expected) and _is_null(fused):
                continue
            
            # Infer type and get match function
            value_type = infer_type(expected)
            match_fn = get_match_function(value_type)
            
            results["total"] += 1
            results["per_attribute"][attr]["total"] += 1
            
            is_match = match_fn(fused, expected)
            
            if is_match:
                results["correct"] += 1
                results["per_attribute"][attr]["correct"] += 1
            else:
                results["mismatches"].append({
                    "id": test_id,
                    "attribute": attr,
                    "type": value_type,
                    "expected": expected,
                    "fused": fused,
                })
    
    return results


# Run evaluation
results = evaluate_fusion(fused_df, test_df)

print(f"\n{'='*60}")
print(f"OVERALL: {results['correct']}/{results['total']} = {results['correct']/results['total']:.1%}" if results['total'] > 0 else "No comparisons made")
print(f"{'='*60}")

Common attributes: ['artist', 'duration', 'label', 'name', 'release-country', 'release-date', 'tracks_track-name']
Fused-only: ['_fusion_sources', '_id', 'genre']
Test-only: ['artist_provenance', 'duration_provenance', 'genre_provenance', 'label_provenance', 'name_provenance', 'release-country_provenance', 'release-date_provenance', 'tracks_provenance']


OVERALL: 92/155 = 59.4%


In [ ]:
# Per-attribute breakdown
print("\nPer-attribute accuracy:")
print("-" * 50)
for attr, stats in sorted(results["per_attribute"].items()):
    if stats["total"] > 0:
        acc = stats["correct"] / stats["total"]
        print(f"{attr:30s}: {stats['correct']:3d}/{stats['total']:3d} = {acc:.1%}")


Per-attribute accuracy:
--------------------------------------------------
artist                        :  18/ 23 = 78.3%
duration                      :   0/ 23 = 0.0%
label                         :  14/ 17 = 82.4%
name                          :  21/ 23 = 91.3%
release-country               :  18/ 23 = 78.3%
release-date                  :  21/ 23 = 91.3%
tracks_track-name             :   0/ 23 = 0.0%


In [ ]:
# Show mismatches
print(f"\nMismatches ({len(results['mismatches'])} total):")
print("-" * 80)

mismatch_df = pd.DataFrame(results["mismatches"])
if not mismatch_df.empty:
    display(mismatch_df.head(50))
else:
    print("No mismatches!")


Mismatches (63 total):
--------------------------------------------------------------------------------


,id,attribute,type,expected,fused
0,mbrainz_6891,tracks_track-name,list,"[Chance, Sabotage, Sabotage, Warrior Poets, Re...","['Chance', 'Sabatoge', 'Warrior Poets', 'Relie..."
1,mbrainz_6891,duration,date,2798,NaN
2,mbrainz_6891,artist,string,Pep Love,P. Love
3,mbrainz_5095,tracks_track-name,list,"[...and Before We Knew It Had Happened At All,...",['...and Before We Knew It Had Happened At All...
4,mbrainz_5095,duration,date,4313,NaN
5,mbrainz_1834,tracks_track-name,list,"[Fields of Yesterday, Soul Collector, Red Ligh...","['Fields Of Yesterday', 'Soul Collector', 'Red..."
6,mbrainz_1834,duration,date,3080,NaN
7,mbrainz_1834,release-country,string,Finland,UK
8,mbrainz_2222,tracks_track-name,list,"[Guns at Dawn, Ratpack]","['Guns at Dawn', 'Ratpack']"
9,mbrainz_2222,duration,numeric,614,NaN


In [ ]:
# Group mismatches by attribute
if not mismatch_df.empty:
    print("\nMismatches by attribute:")
    print(mismatch_df.groupby("attribute").size().sort_values(ascending=False))


Mismatches by attribute:
attribute
duration             23
tracks_track-name    23
artist                5
release-country       5
label                 3
name                  2
release-date          2
dtype: int64


## 6. Export Results

In [ ]:
# Save mismatches for review
if not mismatch_df.empty:
    mismatch_path = FUSION_DIR / "manual_eval_mismatches.csv"
    mismatch_df.to_csv(mismatch_path, index=False)
    print(f"Saved mismatches to {mismatch_path}")

# Save summary
summary = {
    "overall_accuracy": results["correct"] / results["total"] if results["total"] > 0 else 0,
    "total_comparisons": results["total"],
    "correct_comparisons": results["correct"],
    "per_attribute": {
        attr: {
            "accuracy": stats["correct"] / stats["total"] if stats["total"] > 0 else 0,
            "total": stats["total"],
            "correct": stats["correct"],
        }
        for attr, stats in results["per_attribute"].items()
        if stats["total"] > 0
    }
}

summary_path = FUSION_DIR / "manual_eval_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"Saved summary to {summary_path}")

Saved mismatches to fusion/manual_eval_mismatches.csv
Saved summary to fusion/manual_eval_summary.json
